# Stage 2.4 - OOF decoder comparison and evidence report

This notebook aggregates the twenty completed decoder runs from Stage 2.3 and compares the four decoder arms (a 2x2 of {ReLU, PReLU} activation x {plain, residual} skip topology) using paired out-of-fold predictions, decomposing the residual and activation main effects.

The scientific claim is deliberately narrow: performance is conditional on this 71-knee/43-subject cohort, the fold-specific frozen front ends, the matched training protocol, and seed 42. It is not a claim that one original architecture is universally superior or that either model is clinically validated.

## Success criteria

1. Exactly 71 unique OOF knees are present per arm and every knee is paired across all four arms with the same test fold and shared-front-end hash.
2. Bilateral knees are averaged within subject before inference, producing exactly 43 paired subject records.
3. Co-primary endpoints are subject-level macro Dice and macro ASSD in millimetres; no co-primary value is missing or infinite.
4. Two-sided paired Wilcoxon tests use Pratt zero handling, with Holm correction across the co-primary main-effect family (two contrasts x two endpoints).
5. Ten-thousand fold-stratified subject bootstrap resamples provide paired-effect 95% confidence intervals for every contrast.
6. A factor (residual or activation) is preferred only if both Holm-corrected co-primary endpoints are significant in the same direction and the residual x activation interaction is not significant; otherwise the result is `INCONCLUSIVE`.
7. Decoder-family diagonal, residual x activation interaction, pathology interaction, bone-wise, healthy/fractured, topology, parameter, memory, resource, and fold-0 seed-variance results are reported as secondary or descriptive evidence.
8. Regen outputs, if supplied, contain deidentified study IDs only and are labelled illustrative; they cannot select the winning decoder.

## Research-gap interpretation

[Kasten et al. (2020)](https://arxiv.org/abs/2004.00871) and [Lin et al. (2026)](https://pubmed.ncbi.nlm.nih.gov/41840145/) establish direct biplanar knee-bone reconstruction, while [Shakya and Khanal (NeurIPS 2023)](https://papers.neurips.cc/paper_files/paper/2023/hash/412732f172bdd5ad0efde2fafa110700-Abstract-Datasets_and_Benchmarks.html) already benchmark complete biplanar reconstruction systems and explicitly call for disaggregated clinically relevant subgroup reporting. This study therefore addresses a narrower, locally evidenced gap: among the studies reviewed for this project, none isolates matched plain-versus-residual decoding and ReLU-versus-PReLU activation in a 2x2 factorial under a byte-identical pretrained and frozen biplanar knee representation while separately reporting healthy and fractured knees. This is a scoped literature-review claim, not a claim of exhaustive systematic-review novelty.

Dice is paired with ASSD because [Metrics Reloaded (Maier-Hein et al., 2024)](https://doi.org/10.1038/s41592-023-02151-z) recommends problem-aware metric selection, including boundary-sensitive evidence when boundary accuracy is part of the domain interest and explicit handling of empty predictions. Fractured results remain descriptive because only 13 fractured knees are available.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch

STAGE2_SCHEMA="foundation_stage2_v1"
BONES=["femur","tibia","patella","fibula"]
RUN_AGGREGATION=False

def find_project_root(start):
    for candidate in [Path(start).resolve(),*Path(start).resolve().parents]:
        if (candidate/"configs"/"baseline_protocol_v1.json").exists(): return candidate
    raise FileNotFoundError("project root not found")
ROOT=find_project_root(Path.cwd())
SUMMARY_JSON=ROOT/"models"/"decoders"/STAGE2_SCHEMA/"fold_0"/"foundation_summary.json"

In [ ]:
# Complete subject-level OOF comparison (2x2 factorial).
import hashlib
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

RUN_AGGREGATION = False
N_BOOTSTRAP = 10_000
ALPHA = 0.05
SEED = 42
MAIN_SEED = 42
VARIANCE_FOLD = 0
SWEEP_SEEDS = [123, 2024]
ARM_FACTORS = {
    "plain_unet_style":    {"activation": "relu",  "residual": False, "label": "U (ReLU, plain)"},
    "residual_vnet_style": {"activation": "prelu", "residual": True,  "label": "V (PReLU, residual)"},
    "residual_relu_style": {"activation": "relu",  "residual": True,  "label": "ReLU + residual"},
    "plain_prelu_style":   {"activation": "prelu", "residual": False, "label": "PReLU + plain"},
}
ARMS = list(ARM_FACTORS)
ARM_LABELS = {arm: ARM_FACTORS[arm]["label"] for arm in ARM_FACTORS}
DECODER_ROOT = ROOT / "models" / "decoders" / STAGE2_SCHEMA
REPORT_ROOT = ROOT / "reports" / "decoder_comparison" / STAGE2_SCHEMA
PRIMARY_METRICS = ["dice_macro", "assd_mm_macro"]

# Predefined contrasts (registered up front so nothing is chosen post-hoc), letting
# U=plain_unet_style, V=residual_vnet_style, R=residual_relu_style, P=plain_prelu_style.
CONTRASTS = {
    "decoder_family_V_minus_U": lambda a: a["residual_vnet_style"] - a["plain_unet_style"],
    "residual_main": lambda a: 0.5 * ((a["residual_relu_style"] - a["plain_unet_style"]) + (a["residual_vnet_style"] - a["plain_prelu_style"])),
    "activation_main": lambda a: 0.5 * ((a["plain_prelu_style"] - a["plain_unet_style"]) + (a["residual_vnet_style"] - a["residual_relu_style"])),
    "residual_activation_interaction": lambda a: (a["residual_vnet_style"] - a["residual_relu_style"]) - (a["plain_prelu_style"] - a["plain_unet_style"]),
}
CO_PRIMARY = ["residual_main", "activation_main"]  # tested x2 endpoints, Holm-corrected across the four
FACTOR_LABELS = {"residual_main": ("residual", "plain"), "activation_main": ("prelu", "relu")}


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""): digest.update(chunk)
    return digest.hexdigest()


def load_oof_results():
    frames, summaries = [], []
    for fold in range(5):
        for arm in ARMS:
            run_dir = DECODER_ROOT / f"fold_{fold}" / arm; metrics_path = run_dir / "oof_metrics.csv"; summary_path = run_dir / "run_summary.json"; checkpoint_path = run_dir / "best_decoder.pth"
            for required in (metrics_path, summary_path, checkpoint_path):
                if not required.is_file(): raise FileNotFoundError(required)
            summary = json.loads(summary_path.read_text(encoding="utf-8"))
            if summary.get("fold") != fold or summary.get("arm") != arm or not summary.get("success") or summary.get("checkpoint_sha256") != sha256_file(checkpoint_path): raise RuntimeError(f"invalid run summary: fold={fold} arm={arm}")
            frame = pd.read_csv(metrics_path); frame["source_metrics_sha256"] = sha256_file(metrics_path); frames.append(frame); summaries.append(summary)
    oof = pd.concat(frames, ignore_index=True); runs = pd.DataFrame(summaries)
    for arm in ARMS:
        arm_rows = oof[oof.arm.eq(arm)]
        if len(arm_rows) != 71 or arm_rows.sample_id.nunique() != 71: raise RuntimeError(f"{arm} does not contain exactly 71 unique OOF knees")
    pivot_fold = oof.pivot_table(index="sample_id", columns="arm", values="test_fold", aggfunc="first")
    pivot_front = oof.pivot_table(index="sample_id", columns="arm", values="shared_frontend_sha256", aggfunc="first")
    if len(pivot_fold) != 71 or pivot_fold.reindex(columns=ARMS).isna().any().any(): raise RuntimeError("OOF arms are not paired on all 71 knees")
    if not (pivot_fold.nunique(axis=1) == 1).all(): raise RuntimeError("paired knees used different folds across arms")
    if not (pivot_front.nunique(axis=1) == 1).all(): raise RuntimeError("paired knees used different shared front ends across arms")
    if oof[PRIMARY_METRICS].isna().any().any() or not np.isfinite(oof[PRIMARY_METRICS].to_numpy()).all(): raise RuntimeError("missing/non-finite co-primary OOF values")
    return oof, runs


def subject_level_table(oof):
    """Average knees within subject first so bilateral knees do not receive extra weight."""
    metric_columns = [column for column in oof.columns if column.startswith(("dice_", "assd_mm_", "hd95_mm_", "false_bridge_", "empty_prediction_"))]
    subject = oof.groupby(["subject_id", "arm", "test_fold", "cohort"], as_index=False)[metric_columns].mean(numeric_only=True)
    if subject.subject_id.nunique() != 43: raise RuntimeError(f"expected 43 subjects, found {subject.subject_id.nunique()}")
    counts = subject.groupby("arm").subject_id.nunique()
    if not (counts == 43).all(): raise RuntimeError(f"every arm must cover all 43 subjects: {counts.to_dict()}")
    return subject


def arm_matrix(subject, metric):
    wide = subject.pivot_table(index=["subject_id", "test_fold", "cohort"], columns="arm", values=metric).reindex(columns=ARMS)
    if wide.isna().any().any(): raise RuntimeError(f"missing arm values for {metric}")
    return wide.reset_index()


def holm_adjust(p_values):
    order = np.argsort(p_values); adjusted = np.empty(len(p_values), dtype=float); running = 0.0
    for rank, index in enumerate(order):
        value = min(1.0, (len(p_values) - rank) * float(p_values[index])); running = max(running, value); adjusted[index] = running
    return adjusted


def fold_stratified_bootstrap(favourable_effect, folds, n=N_BOOTSTRAP, seed=SEED):
    """Resample subjects within each test fold; positive values already favour the added factor."""
    rng = np.random.default_rng(seed); favourable_effect = np.asarray(favourable_effect, dtype=float); folds = np.asarray(folds)
    sampled_sums = np.zeros(n, dtype=float); subject_count = 0
    for fold in np.unique(folds):
        effects = favourable_effect[folds == fold]
        indices = rng.integers(0, len(effects), size=(n, len(effects)))
        sampled_sums += effects[indices].sum(axis=1); subject_count += len(effects)
    values = sampled_sums / subject_count
    return {"mean_favourable_effect": float(values.mean()), "ci95_low": float(np.percentile(values, 2.5)), "ci95_high": float(np.percentile(values, 97.5)), "resamples": n}


def decide(results):
    interaction_significant = bool((results[results.contrast.eq("residual_activation_interaction")].p_raw < ALPHA).any())
    decisions = {"interaction_significant": interaction_significant}
    for contrast in CO_PRIMARY:
        sub = results[results.contrast.eq(contrast)]; dice = sub[sub.metric.eq("dice_macro")].iloc[0]; assd = sub[sub.metric.eq("assd_mm_macro")].iloc[0]
        both_significant = bool(dice.p_holm < ALPHA and assd.p_holm < ALPHA)
        same_direction = bool(np.sign(dice.favourable_effect_median) == np.sign(assd.favourable_effect_median) and dice.favourable_effect_median != 0)
        added, baseline = FACTOR_LABELS[contrast]
        if both_significant and same_direction and not interaction_significant:
            decisions[contrast] = f"prefers_{added}" if dice.favourable_effect_median > 0 else f"prefers_{baseline}"
        else:
            decisions[contrast] = "INCONCLUSIVE"
    return decisions


def primary_analysis(subject):
    matrices = {metric: arm_matrix(subject, metric) for metric in PRIMARY_METRICS}
    rows = []
    for metric in PRIMARY_METRICS:
        matrix = matrices[metric]; favour_sign = 1.0 if metric == "dice_macro" else -1.0; folds = matrix["test_fold"].to_numpy()
        arm_values = {arm: matrix[arm].to_numpy(float) for arm in ARMS}
        for name, function in CONTRASTS.items():
            effect = function(arm_values); favourable = effect * favour_sign
            test = None if np.allclose(effect, 0.0) else wilcoxon(effect, zero_method="pratt", alternative="two-sided", method="auto")
            rows.append({"metric": metric, "contrast": name, "co_primary": name in CO_PRIMARY, "raw_effect_mean": float(effect.mean()), "favourable_effect_mean": float(favourable.mean()), "favourable_effect_median": float(np.median(favourable)), "wilcoxon_statistic": 0.0 if test is None else float(test.statistic), "p_raw": 1.0 if test is None else float(test.pvalue), **fold_stratified_bootstrap(favourable, folds)})
    results = pd.DataFrame(rows); results["p_holm"] = np.nan
    co_primary = results[results.co_primary]
    results.loc[co_primary.index, "p_holm"] = holm_adjust(co_primary.p_raw.to_numpy())
    return results, decide(results)


def pathology_interaction(subject):
    """Descriptive only (n_fractured = 13, unpaired across cohorts): does an effect differ by pathology?"""
    rows = []
    for metric in PRIMARY_METRICS:
        favour_sign = 1.0 if metric == "dice_macro" else -1.0
        for name in ("residual_main", "decoder_family_V_minus_U"):
            per_cohort = {}
            for cohort in ("healthy", "fractured"):
                matrix = subject[subject.cohort.eq(cohort)].pivot_table(index="subject_id", columns="arm", values=metric).reindex(columns=ARMS)
                if len(matrix) == 0 or matrix.isna().any().any():
                    per_cohort[cohort] = float("nan"); continue
                arm_values = {arm: matrix[arm].to_numpy(float) for arm in ARMS}
                per_cohort[cohort] = float(np.median(CONTRASTS[name](arm_values) * favour_sign))
            rows.append({"metric": metric, "contrast": name, "healthy_median_favourable": per_cohort["healthy"], "fractured_median_favourable": per_cohort["fractured"], "cohort_difference": per_cohort["fractured"] - per_cohort["healthy"], "note": "descriptive_only_n_fractured_13"})
    return pd.DataFrame(rows)


def secondary_tables(oof, subject):
    bone_rows = []
    for bone in BONES:
        for arm in ARMS:
            values = subject[subject.arm.eq(arm)]; bone_rows.append({"bone": bone, "arm": arm, "arm_label": ARM_LABELS[arm], "subjects": len(values), "dice_mean": float(values[f"dice_{bone}"].mean()), "assd_mm_mean": float(values[f"assd_mm_{bone}"].mean())})
    cohort = subject.groupby(["cohort", "arm"], as_index=False)[PRIMARY_METRICS].agg(["count", "mean", "median"]).reset_index()
    topology_columns = [column for column in subject.columns if column.startswith(("false_bridge_", "empty_prediction_"))]
    topology = subject.groupby("arm", as_index=False)[topology_columns].mean(numeric_only=True)
    return pd.DataFrame(bone_rows), cohort, topology


def load_variance_band():
    """Fold-0 across-seed dispersion of subject-macro metrics, so an effect can be judged against
    training-run variability (seed 42 from main CV plus the sweep seeds)."""
    sweep_root = DECODER_ROOT / "seed_sweep"
    rows = []
    for arm in ARMS:
        per_seed = {}
        main_metrics = DECODER_ROOT / f"fold_{VARIANCE_FOLD}" / arm / "oof_metrics.csv"
        if main_metrics.is_file():
            per_seed[MAIN_SEED] = pd.read_csv(main_metrics)
        for seed in SWEEP_SEEDS:
            sweep_metrics = sweep_root / f"fold_{VARIANCE_FOLD}" / arm / f"seed_{seed}" / "oof_metrics.csv"
            if sweep_metrics.is_file():
                per_seed[seed] = pd.read_csv(sweep_metrics)
        for seed, frame in per_seed.items():
            subject_means = frame.groupby("subject_id")[PRIMARY_METRICS].mean(numeric_only=True)
            rows.append({"arm": arm, "seed": seed, "n_subjects": int(len(subject_means)), **{f"{metric}_mean": float(subject_means[metric].mean()) for metric in PRIMARY_METRICS}})
    band = pd.DataFrame(rows)
    if band.empty:
        return {"status": "NOT_PROVIDED"}, band
    dispersion = band.groupby("arm")[[f"{metric}_mean" for metric in PRIMARY_METRICS]].agg(["mean", "std", "count"])
    dispersion.columns = ["_".join(column) for column in dispersion.columns]
    return {"status": "AVAILABLE", "seeds": sorted(int(seed) for seed in band.seed.unique()), "arms": len(band.arm.unique())}, band.merge(dispersion.reset_index(), on="arm")


def resource_table(runs):
    rows = []
    for record in runs.to_dict("records"):
        resource = record.get("resource_usage", {}); architecture = record.get("architecture", {})
        rows.append({"fold": record["fold"], "arm": record["arm"], "arm_label": record.get("arm_label", ARM_LABELS[record["arm"]]), "seed": record.get("seed"), "logit_resolution": record.get("logit_resolution"), "parameters": architecture.get("parameters"), "peak_gpu_bytes": resource.get("peak_gpu_bytes"), "gpu_headroom_fraction": resource.get("gpu_headroom_fraction"), "wall_seconds": resource.get("wall_seconds")})
    return pd.DataFrame(rows)


def save_figures(bone, subject):
    REPORT_ROOT.mkdir(parents=True, exist_ok=True)
    labels = [ARM_LABELS[arm] for arm in ARMS]
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for axis, metric, title in zip(axes, PRIMARY_METRICS, ["Subject macro Dice", "Subject macro ASSD (mm)"]):
        data = [subject[subject.arm.eq(arm)][metric].to_numpy() for arm in ARMS]; axis.boxplot(data, labels=labels, showmeans=True); axis.set_title(title); axis.grid(alpha=0.25); axis.tick_params(axis="x", rotation=20)
    figure.tight_layout(); figure.savefig(REPORT_ROOT / "co_primary_boxplots.png", dpi=180); plt.close(figure)
    pivot_dice = bone.pivot(index="bone", columns="arm", values="dice_mean").loc[BONES]; pivot_assd = bone.pivot(index="bone", columns="arm", values="assd_mm_mean").loc[BONES]
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5)); pivot_dice.plot(kind="bar", ax=axes[0]); pivot_assd.plot(kind="bar", ax=axes[1]); axes[0].set_title("Per-bone Dice"); axes[1].set_title("Per-bone ASSD (mm)")
    for axis in axes: axis.grid(axis="y", alpha=0.25); axis.tick_params(axis="x", rotation=0)
    figure.tight_layout(); figure.savefig(REPORT_ROOT / "per_bone_comparison.png", dpi=180); plt.close(figure)


def validate_regen_illustrations():
    path = REPORT_ROOT / "regen_illustrative_manifest.csv"
    if not path.is_file(): return {"status": "NOT_PROVIDED", "used_for_winner": False}
    frame = pd.read_csv(path); forbidden = {"patient_name", "patient_id", "filename", "folder_name"}
    if forbidden & set(column.casefold() for column in frame.columns): raise RuntimeError("Regen illustration manifest contains a forbidden PII-bearing column")
    if "study_id" not in frame.columns or frame.study_id.astype(str).str.strip().eq("").any(): raise RuntimeError("Regen illustration manifest requires non-empty study_id")
    return {"status": "ILLUSTRATIVE_ONLY", "rows": len(frame), "used_for_winner": False, "sha256": sha256_file(path)}


def run_comparison():
    REPORT_ROOT.mkdir(parents=True, exist_ok=True); oof, runs = load_oof_results(); subject = subject_level_table(oof); primary, decisions = primary_analysis(subject); bone, cohort, topology = secondary_tables(oof, subject); pathology = pathology_interaction(subject); resources = resource_table(runs); variance_status, variance = load_variance_band(); regen = validate_regen_illustrations()
    oof.to_csv(REPORT_ROOT / "oof_all_knees.csv", index=False); subject.to_csv(REPORT_ROOT / "subject_level_metrics.csv", index=False); primary.to_csv(REPORT_ROOT / "co_primary_results.csv", index=False); bone.to_csv(REPORT_ROOT / "per_bone_results.csv", index=False); cohort.to_csv(REPORT_ROOT / "cohort_results.csv", index=False); topology.to_csv(REPORT_ROOT / "topology_results.csv", index=False); pathology.to_csv(REPORT_ROOT / "pathology_interaction_results.csv", index=False); resources.to_csv(REPORT_ROOT / "resource_results.csv", index=False)
    if not variance.empty: variance.to_csv(REPORT_ROOT / "variance_band_results.csv", index=False)
    save_figures(bone, subject)
    summary = {"schema_version": STAGE2_SCHEMA, "analysis": "controlled_decoder_factorial_oof_comparison", "seed": SEED, "arms": ARMS, "knees_per_arm": {arm: int((oof.arm == arm).sum()) for arm in ARMS}, "paired_subjects": int(subject.subject_id.nunique()), "co_primary_metrics": PRIMARY_METRICS, "co_primary_contrasts": CO_PRIMARY, "decision_rule": "both Holm-corrected co-primary p-values < 0.05 in the same direction on both endpoints, with a non-significant residual x activation interaction", "decisions": decisions, "contrasts": primary.to_dict("records"), "pathology_interaction": pathology.to_dict("records"), "variance_band": variance_status, "fractured_subjects": int(subject[subject.cohort.eq("fractured")].subject_id.nunique()), "healthy_subjects": int(subject[subject.cohort.eq("healthy")].subject_id.nunique()), "regen": regen, "limitations": ["one training seed per fold (fold-0 variance band only)", "fractured subgroup is descriptive", "fixed-front-end conditional inference", "arms differ in residual topology and activation; the 2x2 estimates main effects but power is limited", "logits computed at 128^3 then upsampled, so ASSD is bounded by effective spacing, not target spacing", "no clinical-utility claim"], "success": len(oof) == 71 * len(ARMS) and subject.subject_id.nunique() == 43 and not primary[primary.co_primary].p_holm.isna().any()}
    (REPORT_ROOT / "comparison_summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    if not summary["success"]: raise RuntimeError("comparison success criteria failed")
    return summary, primary, bone, cohort, topology, pathology, resources, variance

In [ ]:
if RUN_AGGREGATION:
    summary, primary, bone, cohort, topology, pathology, resources, variance = run_comparison()
    display(primary); display(bone); display(cohort); display(topology); display(pathology); display(resources)
    if not variance.empty: display(variance)
    print("co-primary decisions:", summary["decisions"])
else:
    print("Definitions loaded. Set RUN_AGGREGATION=True only after all twenty Stage 2.3 run summaries report success.")